In [ ]:
THRESHOLD = 0.85      # limiar de similaridade cosseno
TOP_K = 50            # número de vizinhos mais próximos buscados por comentário
RANDOM_SEED = 42

In [ ]:
import os
import hashlib
import random
import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
import faiss
import chromadb
from chromadb import Settings
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from transformers import pipeline

In [ ]:
load_dotenv()

VIDEO_ID = os.getenv("VIDEO_ID1")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"VIDEO_ID: {VIDEO_ID} | THRESHOLD: {THRESHOLD} | TOP_K: {TOP_K}")


In [ ]:
csv_path = os.getenv("CLEAN_COMMENTS_PATH")
data_path = os.getenv("DATA_PATH")
chroma_collection = os.getenv("CHROMA_COLLECTION")

# pasta de saída = pasta pai do DATA_PATH (data/chromadb/ → data/)
output_path = os.path.dirname(data_path.rstrip("/"))

df = pd.read_csv(csv_path)
df['comment'] = df['comment'].fillna("").astype(str)
texts = df['comment'].tolist()
ids_md5 = [hashlib.md5(t.encode()).hexdigest() for t in texts]
print(f"Total de comentários no CSV: {len(texts)}")

# conectar ao ChromaDB
client = chromadb.PersistentClient(path=data_path, settings=Settings(anonymized_telemetry=False))
collection = client.get_collection(name=chroma_collection)
print(f"ChromaDB conectado. Coleção '{chroma_collection}' tem {collection.count()} documentos.")

# recuperar embeddings em batches de 5000
embeddings_dict = {}
batch_size = 5000
for i in range(0, len(ids_md5), batch_size):
    batch_ids = ids_md5[i:i+batch_size]
    result = collection.get(ids=batch_ids, include=["embeddings"])
    for doc_id, emb in zip(result["ids"], result["embeddings"]):
        embeddings_dict[doc_id] = emb
    if i % 20000 == 0:
        print(f"  Carregados {len(embeddings_dict)} embeddings...")

# manter apenas comentários com embedding encontrado
indices_validos = [i for i, doc_id in enumerate(ids_md5) if doc_id in embeddings_dict]
texts = [texts[i] for i in indices_validos]
ids_md5 = [ids_md5[i] for i in indices_validos]
embeddings_array = np.array([embeddings_dict[doc_id] for doc_id in ids_md5], dtype=np.float32)

print(f"Embeddings carregados: {embeddings_array.shape}")
print(f"Saídas serão salvas em: {output_path}")


In [ ]:
# índice FAISS
# normalização L2: produto interno entre vetores normalizados == similaridade cosseno
faiss.normalize_L2(embeddings_array)

dim = embeddings_array.shape[1]
index = faiss.IndexFlatIP(dim)  # produto interno (= cosseno após normalização)
index.add(embeddings_array)
print(f"Índice FAISS construído com {index.ntotal} vetores de dimensão {dim}")

# buscar top-K vizinhos para todos os comentários de uma vez
print(f"Buscando top-{TOP_K} vizinhos para cada comentário...")
similarities, indices_nn = index.search(embeddings_array, TOP_K + 1)  # +1 pois o próprio comentário aparece
print("Busca concluída.")

In [ ]:
# construção de arestas
edges = []  # lista de (i, j, similaridade)

for i in range(len(texts)):
    for rank in range(1, TOP_K + 1):  # rank 0 é o próprio comentário
        j = int(indices_nn[i, rank])
        sim = float(similarities[i, rank])
        # apenas pares (i, j) com j > i para evitar duplicatas
        if j > i and sim >= THRESHOLD:
            edges.append((i, j, sim))

print(f"Arestas acima do threshold ({THRESHOLD}): {len(edges)}")

In [ ]:
G = nx.Graph()

for i, (doc_id, text) in enumerate(zip(ids_md5, texts)):
    G.add_node(i, doc_id=doc_id, text=text)

for i, j, sim in edges:
    G.add_edge(i, j, weight=sim)

print(f"Grafo inicial: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

In [ ]:
# remover nós isolados e extrair componente gigante
G.remove_nodes_from(list(nx.isolates(G)))

if len(G) > 0:
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()

print(f"Após filtragem: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

In [ ]:
# detecção de comunidades com Leiden
nodes = list(G.nodes())
node_index = {n: i for i, n in enumerate(nodes)}

ig_graph = ig.Graph(n=len(nodes))
ig_edges = [(node_index[u], node_index[v]) for u, v in G.edges()]
ig_weights = [G[u][v]["weight"] for u, v in G.edges()]
ig_graph.add_edges(ig_edges)

partition = leidenalg.find_partition(
    ig_graph,
    leidenalg.ModularityVertexPartition,
    weights=ig_weights,
    seed=RANDOM_SEED
)

partition_dict = {}
for community_id, members in enumerate(partition):
    for member_idx in members:
        partition_dict[nodes[member_idx]] = community_id

mod = partition.modularity
print(f"Modularidade (Leiden): {mod:.4f}")
print(f"Número de comunidades: {len(partition)}")

for node in G.nodes():
    G.nodes[node]['community_id'] = partition_dict[node]

In [ ]:
# rotulação das comunidades com LLM

print("Carregando modelos de rotulação...")
gerador = pipeline("text2text-generation", model="google/flan-t5-base")
classificador = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

deg_cent = nx.degree_centrality(G)

df_nodes = pd.DataFrame({
    "node": nodes,
    "text": [G.nodes[n]["text"] for n in nodes],
    "degree_centrality": [deg_cent[n] for n in nodes],
    "community_id": [partition_dict[n] for n in nodes]
})

comunidades_ids = sorted(df_nodes["community_id"].unique())
rotulos_gerados = {}

for com_id in comunidades_ids:
    top_comentarios = (
        df_nodes[df_nodes["community_id"] == com_id]
        .sort_values("degree_centrality", ascending=False)
        .head(20)["text"]
        .tolist()
    )
    comentarios_str = " | ".join(top_comentarios[:5])

    # Passo A: Text Generation — rótulo livre
    prompt = f"These are comments from a Brazilian political debate. Identify the central theme in up to 5 words: {comentarios_str}"
    resultado = gerador(prompt, max_new_tokens=20)[0]["generated_text"]
    rotulo_livre = resultado.strip()

    # Passo B: Zero-Shot Classification — validação
    hipoteses = [rotulo_livre, "ataques políticos", "pauta econômica", "mobilização ideológica", "discurso moralizante"]
    resultado_zs = classificador(comentarios_str, candidate_labels=hipoteses)
    rotulo_final = resultado_zs["labels"][0]

    rotulos_gerados[com_id] = rotulo_final
    print(f"Comunidade {com_id}: '{rotulo_livre}' → validado como '{rotulo_final}'")

print("\nRótulos finais:")
for com_id, rotulo in rotulos_gerados.items():
    n = len(df_nodes[df_nodes["community_id"] == com_id])
    print(f"  {com_id}: {rotulo} ({n} nós)")

for node in G.nodes():
    G.nodes[node]['community_label'] = rotulos_gerados.get(partition_dict[node], str(partition_dict[node]))

In [ ]:
# análise exploratória
print("=== Métricas da Rede de Similaridade (FAISS) ===")
print(f"Nós (comentários): {G.number_of_nodes()}")
print(f"Arestas: {G.number_of_edges()}")
print(f"Densidade: {nx.density(G):.6f}")
print(f"Número de comunidades: {len(rotulos_gerados)}")
print(f"Modularidade (Leiden): {mod:.4f}")

print("\n=== Comunidades ===")
for com_id, rotulo in rotulos_gerados.items():
    n = len(df_nodes[df_nodes["community_id"] == com_id])
    print(f"  [{com_id}] {rotulo}: {n} comentários ({100*n/G.number_of_nodes():.1f}%)")

degrees = [d for _, d in G.degree()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(degrees, bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel("Grau")
ax.set_ylabel("Frequência")
ax.set_title(f"Distribuição de Grau — Rede de Similaridade FAISS ({VIDEO_ID})")
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"distribuicao_grau_similaridade_faiss_{VIDEO_ID}.png"), dpi=150)
plt.show()


In [ ]:
# GEXF para Gephi
caminho_gexf = os.path.join(output_path, f"grafo_similaridade_faiss_{VIDEO_ID}.gexf")
nx.write_gexf(G, caminho_gexf)
print(f"Grafo exportado: {caminho_gexf}")

df_resumo = df_nodes.copy()
df_resumo["community_label"] = df_resumo["community_id"].map(rotulos_gerados)

caminho_csv = os.path.join(output_path, f"comunidades_similaridade_faiss_{VIDEO_ID}.csv")
df_resumo[["node", "text", "degree_centrality", "community_id", "community_label"]].to_csv(
    caminho_csv, index=False, encoding='utf-8-sig'
)
print(f"Tabela de comunidades exportada: {caminho_csv}")
